# Period 3: Orders

Practise bench for the **orders table**, the **`/order` route** and **`/my-orders`**.

**How this notebook works**
1. **Practise here first.** Write and test each piece in the cells below. A cell that says **TODO** is yours to write.
2. Click a cell and press **Shift+Enter** to run it. Run the cells **in order, from the top**.
3. A check cell prints **✓** when you are right, or **✗** and a hint when you are not. Fix the cell above it and run both again.
4. When every check shows ✓, **copy your code into the real app files** (the last section tells you where).

If a cell shows a red error such as `no such table`, you skipped a TODO cell above it. Run that one first.

The database in this notebook is a **practice copy that lives in memory**. It is not your real `ais.db`, so you cannot break anything.

**SQL cells.** A cell whose first line is `%%sql` holds plain SQL. The `%%sql` line is not part of your SQL. A cell that makes or fills a table shows an empty box under it: that is normal.

In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")   # a practice database that lives in memory
db.row_factory = sqlite3.Row       # so each row can be read like a dictionary: row["name"]

# Turn on SQL cells and point them at the practice database above.
# (Given. No comments go after a % line: they would be read as part of it.)
%load_ext sql
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = False
%sql db

def check(ok, good, hint):
    """Print a tick or a cross. (Given.)"""
    if ok:
        print("✓ " + good)
    else:
        print("✗ " + hint)


# A practice menu and users, so you can place orders. (Given.)
db.executescript("""
CREATE TABLE menu (id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT NOT NULL, price REAL NOT NULL);
INSERT INTO menu (name, price) VALUES ('Nasi lemak', 3.50);
INSERT INTO menu (name, price) VALUES ('Chicken schnitzel wrap', 4.50);
INSERT INTO menu (name, price) VALUES ('Singapore laksa', 4.00);
INSERT INTO menu (name, price) VALUES ('Vegetable sushi (6 pc)', 3.80);
INSERT INTO menu (name, price) VALUES ('Aussie beef pie', 3.20);
CREATE TABLE users (id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT UNIQUE NOT NULL, password TEXT NOT NULL, role TEXT NOT NULL);
INSERT INTO users (username, password, role) VALUES ('aisha', 'sunflower22', 'user');
INSERT INTO users (username, password, role) VALUES ('ben', 'basketball7', 'user');
""")

In [ ]:
%%sql
SELECT id, name, price FROM menu

## 1. The orders table

One row is one order of one item. The columns are `id`, `username`, `item`, `price` and `created`. `id` counts up by itself and the other four are text or a number that cannot be empty (`price` is a decimal number).

### TODO 1: create the table

Write your SQL in the cell below. Delete the placeholder line when you have written your own.

In [ ]:
%%sql
-- TODO 1: write CREATE TABLE IF NOT EXISTS orders (...) here.
-- Delete the next line when you have written your SQL.
SELECT 'write your SQL here' AS todo;

In [ ]:
cols = [r["name"] for r in db.execute("PRAGMA table_info(orders)")]
check(cols == ["id", "username", "item", "price", "created"], "The orders table has the right columns.",
      "Expected the columns id, username, item, price, created. Found: " + str(cols) + ".")

## 2. Save an order

The function looks up the item by its id with a **`?` placeholder**: the value is passed separately, in a tuple, so it is treated as data. You use the same idea to save the order.

### TODO 2: INSERT the order

Add an `INSERT` with **four `?` placeholders** (`username`, `item`, `price`, `created`) and pass the four values as a tuple. Then `commit()`.

In [ ]:
from datetime import datetime

def place_order(username, item_id):
    item = db.execute("SELECT name, price FROM menu WHERE id = ?", (item_id,)).fetchone()
    now = datetime.now().strftime("%Y-%m-%d %H:%M")
    # TODO 2: INSERT one row into orders using four ? placeholders, then commit

In [ ]:
place_order("aisha", 3)
place_order("aisha", 5)
place_order("ben", 1)
n = db.execute("SELECT COUNT(*) FROM orders").fetchone()[0] if cols else 0
check(n == 3, "Three orders are saved.", "Expected 3 saved orders but found " + str(n) + ". Did you commit()?")
row = db.execute("SELECT username, item, price FROM orders WHERE id = 1").fetchone() if n else None
check(row is not None and row["username"] == "aisha" and row["item"] == "Singapore laksa" and row["price"] == 4.0,
      "The first order stored aisha, Singapore laksa and 4.0.",
      "The first order should store aisha, Singapore laksa and 4.0. Got: " + (str(tuple(row)) if row else "nothing") + ".")

In [ ]:
%%sql
SELECT * FROM orders

## 3. Read back only one person's orders

`WHERE username = ?` returns only that person's rows, so two students never see each other's orders. `ORDER BY id DESC` puts the newest first. `sum(...)` adds up the prices for a total.

### TODO 3: finish `my_orders`

In [ ]:
def my_orders(username):
    # TODO 3a: SELECT item, price, created FROM orders WHERE username = ?  (newest first)
    rows = []
    # TODO 3b: work out the total price of those rows
    total = 0.0
    return rows, total

In [ ]:
rows_a, total_a = my_orders("aisha")
check([r["item"] for r in rows_a] == ["Aussie beef pie", "Singapore laksa"], "aisha sees her two orders, newest first.",
      "aisha should see Aussie beef pie then Singapore laksa. You got: " + str([r["item"] for r in rows_a]) + ".")
check(abs(total_a - 7.2) < 0.001, "aisha's total is 7.2.", "aisha's total should be 7.2 but is " + str(total_a) + ".")
rows_b, total_b = my_orders("ben")
check([r["item"] for r in rows_b] == ["Nasi lemak"], "ben only sees his own order.", "ben should see only Nasi lemak.")
rows_n, total_n = my_orders("nobody")
check(rows_n == [] and total_n == 0, "Someone with no orders sees nothing, total 0.", "A person with no orders should give an empty list and a total of 0.")

## 4. Show the orders on the page

### TODO 4: the loop

Replace `(write your loop here)` with a Jinja loop over `orders`: one `<li>` per order showing `o.item`, `o.price` and `o.created`.

In [ ]:
from jinja2 import Template

page = Template("""
<ul class="order-list">
  (write your loop here)
</ul>
<p>Total: {{ total }}</p>
""")
html = page.render(orders=rows_a, total=total_a)
print(html)

In [ ]:
check(html.count("<li") == len(rows_a), "One <li> for each of aisha's orders.", "Expected " + str(len(rows_a)) + " <li> items, found " + str(html.count("<li")) + ".")
check(all(r["item"] in html for r in rows_a), "Every item name is on the page.", "Use {{ o.item }} inside the loop.")

## 5. Stop and think

You have now built a query **two ways**:

- **Way 1: join the text.** `"... WHERE username = '" + username + "'"` (your login)
- **Way 2: placeholders.** `"... WHERE username = ?", (username,)` (your orders)

For a normal name, both give the same answer:

In [ ]:
way1 = db.execute("SELECT item FROM orders WHERE username = '" + "aisha" + "'").fetchall()
way2 = db.execute("SELECT item FROM orders WHERE username = ?", ("aisha",)).fetchall()
print(len(way1), len(way2))

**Both work. Which do you think is safer, and why?** Write your prediction here (double-click to edit). **Period 4 gives you the answer.**

My prediction: ...

## 6. Move it into the real app

| What | From this notebook | Into |
| --- | --- | --- |
| The orders table | your TODO 1 SQL | `setup_db.py`, in the **second** pair of triple quotes. Then run `python3 setup_db.py` again |
| Saving | your TODO 2 `INSERT` and `commit()` | `main.py`, in `/order` (replace the TODO and the last return line, and keep `return redirect(url_for("my_orders"))`) |
| Reading | your TODO 3 query and total | `main.py`, in `/my-orders` (replace the TODO and the three lines below it) |
| The list | your TODO 4 loop | `templates/my_orders.html`, in place of the TODO note |

Then run the check cell below. It places orders as aisha and ben in a practice copy of your data.

In [ ]:
import os, shutil, tempfile, importlib

def load_app():
    """Load your main.py. (Given: you do not need to read this.)"""
    import main
    importlib.reload(main)
    return main

def app_with_copy_of_db(main):
    """Point the app at a COPY of your ais.db, so these checks never change your real data."""
    tmp = os.path.join(tempfile.mkdtemp(), "check.db")
    if os.path.exists("ais.db"):
        shutil.copy("ais.db", tmp)
    main.DB_PATH = tmp
    main.init_db()          # makes the users table if it is missing
    return main

In [ ]:
def check_my_orders():
    """Check your real /order and /my-orders. (Given: you do not need to read this.)"""
    main = app_with_copy_of_db(load_app())
    names = []
    try:
        names = [r[0] for r in sqlite3.connect(main.DB_PATH).execute("SELECT name FROM sqlite_master WHERE type='table'")]
    except sqlite3.Error:
        pass
    if "orders" not in names or "menu" not in names:
        check(False, "", "Your ais.db needs a menu table and an orders table. Check setup_db.py and run python3 setup_db.py.")
        return

    first = sqlite3.connect(main.DB_PATH).execute("SELECT id, name, price FROM menu ORDER BY id").fetchall()
    id1, name1, price1 = first[0]
    id2, name2, price2 = first[1]

    a = main.app.test_client()
    a.set_cookie("uid", "aisha")
    a.post("/order", data={"item": str(id1)})
    a.post("/order", data={"item": str(id2)})
    page = a.get("/my-orders").get_data(as_text=True)
    check(name1 in page and name2 in page, "aisha's two orders show on /my-orders.",
          "After ordering " + name1 + " and " + name2 + " as aisha, /my-orders does not list both. Check /order (INSERT and commit) and /my-orders in main.py.")
    want = "%.2f" % (price1 + price2)
    check(want in page, "The total on /my-orders is " + want + ".",
          "The total on /my-orders should be " + want + ". Check the sum in main.py.")

    b = main.app.test_client()
    b.set_cookie("uid", "ben")
    page_b = b.get("/my-orders").get_data(as_text=True)
    check(name1 not in page_b and name2 not in page_b, "ben does not see aisha's orders.",
          "ben can see aisha's orders. Check that /my-orders uses WHERE username = ? with the signed-in user.")

check_my_orders()